In [2]:
from itertools import combinations
from tqdm import tqdm
import numpy as np
import os

D_epoch = {'0': '11', '1': '16', '2': '21', '3': '26', '4': '31', '5': '36'}
D_activ = {'0': "gelu", '1': "relu", '2': "silu", '3': "leakyrelu", '4': "sigmoid", '5': "tanh"}

for m in [0, 1, 2]:
    All = []
    digits = list(range(10))
    for k1 in tqdm(range(2, 10)):
        S1 = list(combinations(digits, k1))
        for pair1 in S1:
            for k2 in range(2, 10):
                S2 = list(combinations(digits, k2))
                for pair2 in S2:
                    if pair1 != pair2 and len(set(pair1) & set(pair2)) == m:
                        if [list(pair1), list(pair2)] not in All:
                            All.append([list(pair1), list(pair2)])
    for activ_key in range(6):
        activ = str(activ_key)
        for epoch_key in range(6):
            epoch = str(epoch_key)
            results_path = f"./data/Scenario/overlapping_m{m}_epoch{epoch}_activ{activ}/"
            if not os.path.isdir(results_path):
                os.mkdir(results_path)
            train_pairs = []
            val_pairs = []
            test_pairs = []
            for element in All:
                pair1, pair2 = element
                full_pair = [pair1, pair2, int(epoch), int(activ)]  # Add epoch and activ as integers
                if 0 in pair1 or 0 in pair2:
                    test_pairs.append(full_pair)
                elif len(pair1) == len(pair2):
                    val_pairs.append(full_pair)
                else:
                    train_pairs.append(full_pair)
            if len(train_pairs) > 0:
                np.save(os.path.join(results_path, "train_pairs.npy"), np.array(train_pairs, dtype=object))
                if len(val_pairs) > 0:
                    np.save(os.path.join(results_path, "val_pairs.npy"), np.array(val_pairs, dtype=object))
                if len(test_pairs) > 0:
                    np.save(os.path.join(results_path, "test_pairs.npy"), np.array(test_pairs, dtype=object))

100%|| 8/8 [25:39<00:00, 192.41s/it]


In [5]:
import pandas as pd
import numpy as np
import torch
import os
from tqdm import tqdm
import gc

df = pd.read_csv("./data/Merged zoo.csv")
params_cols = list(df.columns[17:-2])  # Match columns used in CustomDataset
D_activ = {'0': "gelu", '1': "relu", '2': "silu", '3': "leakyrelu", '4': "sigmoid", '5': "tanh"}
D_epoch = {'0': '11', '1': '16', '2': '21', '3': '26', '4': '31', '5': '36'}

print("Converting .npy files to .pth files for all overlapping scenarios...")

total_scenarios = 0
processed_scenarios = 0

# Process in chunks to manage memory
def process_scenario_in_chunks(m, activ_key, epoch_key, chunk_size=100):
    """Process a single scenario in chunks to save memory"""
    results_path = f"./data/Scenario/overlapping_m{m}_epoch{epoch_key}_activ{activ_key}/"
    if not os.path.exists(results_path):
        return 0, 0, 0
    
    activ_str = D_activ[str(activ_key)]
    epoch_str = D_epoch[str(epoch_key)]
    scenario_name = f"overlapping_m{m}_epoch{epoch_key}_activ{activ_key}"
    
    train_batches = 0
    val_batches = 0
    test_batches = 0
    
    # Process train pairs in chunks
    train_file = os.path.join(results_path, "train_pairs.npy")
    if os.path.exists(train_file):
        train_pairs = np.load(train_file, allow_pickle=True)
        total_pairs = len(train_pairs)
        
        print(f"Processing {scenario_name} - {total_pairs} train pairs...")
        
        # Process in chunks
        for chunk_start in tqdm(range(0, total_pairs, chunk_size), desc=f"Train chunks m{m}e{epoch_key}a{activ_key}"):
            chunk_end = min(chunk_start + chunk_size, total_pairs)
            chunk_pairs = train_pairs[chunk_start:chunk_end]
            
            L_loaded = []
            L_ACC = []
            L_indexes = []
            
            for pair in chunk_pairs:
                pair1, pair2, epoch_int, activ_int = pair
                label1_str = str(pair1)
                label2_str = str(pair2)
                tgt_label_list = sorted(list(set(pair1 + pair2)))
                tgt_label_str = str(tgt_label_list)
                
                # FIXED: Use dictionary values for queries
                row1 = df[(df["label"] == label1_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
                row2 = df[(df["label"] == label2_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
                row3 = df[(df["label"] == tgt_label_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
                
                if len(row1) == 1 and len(row2) == 1 and len(row3) == 1:
                    Stream1 = torch.tensor(row1[params_cols].values[0], dtype=torch.float32)
                    ACC1 = row1["Accuracy"].values[0]
                    ind1 = row1.index[0]
                    Stream2 = torch.tensor(row2[params_cols].values[0], dtype=torch.float32)
                    ACC2 = row2["Accuracy"].values[0]
                    ind2 = row2.index[0]
                    target = torch.tensor(row3[params_cols].values[0], dtype=torch.float32)
                    ACC3 = row3["Accuracy"].values[0]
                    ind3 = row3.index[0]
                    loaded = torch.stack([Stream1, Stream2, target], dim=0)
                    L_ACC.append([ACC1, ACC2, ACC3])
                    L_indexes.append([ind1, ind2, ind3])
                    L_loaded.append(loaded)
            
            # Save chunk to temporary file
            if L_loaded:
                chunk_file = os.path.join(results_path, f"train_chunk_{chunk_start//chunk_size}.pt")
                torch.save({"loaded_list": L_loaded, "L_ACC_list": L_ACC, "L_indexes_list": L_indexes}, chunk_file)
                train_batches += len(L_loaded)
            
            # Clear memory
            del L_loaded, L_ACC, L_indexes
            gc.collect()
        
        # Merge all chunks into final file
        chunk_files = [f for f in os.listdir(results_path) if f.startswith("train_chunk_")]
        num_chunks = len(chunk_files)
        print(f"Merging {num_chunks} train chunks...")
        
        final_train_loaded = []
        final_train_acc = []
        final_train_indexes = []
        
        for chunk_file in chunk_files:
            chunk_path = os.path.join(results_path, chunk_file)
            chunk_data = torch.load(chunk_path, map_location='cpu',weights_only=False)
            final_train_loaded.extend(chunk_data['loaded_list'])
            final_train_acc.extend(chunk_data['L_ACC_list'])
            final_train_indexes.extend(chunk_data['L_indexes_list'])
            os.remove(chunk_path)  # Remove temporary chunk file
        
        if final_train_loaded:
            torch.save({"loaded_list": final_train_loaded, "L_ACC_list": final_train_acc, "L_indexes_list": final_train_indexes}, 
                      os.path.join(results_path, "train_batches.pt"))
        
        del final_train_loaded, final_train_acc, final_train_indexes
        gc.collect()
    
    # Process val pairs (smaller, can do in one go)
    val_file = os.path.join(results_path, "val_pairs.npy")
    if os.path.exists(val_file):
        val_pairs = np.load(val_file, allow_pickle=True)
        L_loaded_val = []
        L_ACC_val = []
        L_indexes_val = []
        
        for pair in val_pairs:
            pair1, pair2, epoch_int, activ_int = pair
            label1_str = str(pair1)
            label2_str = str(pair2)
            tgt_label_list = sorted(list(set(pair1 + pair2)))
            tgt_label_str = str(tgt_label_list)
            
            # FIXED: Use dictionary values for queries
            row1 = df[(df["label"] == label1_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            row2 = df[(df["label"] == label2_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            row3 = df[(df["label"] == tgt_label_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            
            if len(row1) == 1 and len(row2) == 1 and len(row3) == 1:
                Stream1 = torch.tensor(row1[params_cols].values[0], dtype=torch.float32)
                ACC1 = row1["Accuracy"].values[0]
                ind1 = row1.index[0]
                Stream2 = torch.tensor(row2[params_cols].values[0], dtype=torch.float32)
                ACC2 = row2["Accuracy"].values[0]
                ind2 = row2.index[0]
                target = torch.tensor(row3[params_cols].values[0], dtype=torch.float32)
                ACC3 = row3["Accuracy"].values[0]
                ind3 = row3.index[0]
                loaded = torch.stack([Stream1, Stream2, target], dim=0)
                L_ACC_val.append([ACC1, ACC2, ACC3])
                L_indexes_val.append([ind1, ind2, ind3])
                L_loaded_val.append(loaded)
        
        if L_loaded_val:
            torch.save({"loaded_list": L_loaded_val, "L_ACC_list": L_ACC_val, "L_indexes_list": L_indexes_val}, 
                      os.path.join(results_path, "val_batches.pt"))
            val_batches = len(L_loaded_val)
        
        del L_loaded_val, L_ACC_val, L_indexes_val
        gc.collect()
    
    # Process test pairs (smaller, can do in one go)
    test_file = os.path.join(results_path, "test_pairs.npy")
    if os.path.exists(test_file):
        test_pairs = np.load(test_file, allow_pickle=True)
        L_loaded_test = []
        L_ACC_test = []
        L_indexes_test = []
        
        for pair in test_pairs:
            pair1, pair2, epoch_int, activ_int = pair
            label1_str = str(pair1)
            label2_str = str(pair2)
            tgt_label_list = sorted(list(set(pair1 + pair2)))
            tgt_label_str = str(tgt_label_list)
            
            # FIXED: Use dictionary values for queries
            row1 = df[(df["label"] == label1_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            row2 = df[(df["label"] == label2_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            row3 = df[(df["label"] == tgt_label_str) & (df["epoch"] == int(epoch_str)) & (df[activ_str] == 1.0)]
            
            if len(row1) == 1 and len(row2) == 1 and len(row3) == 1:
                Stream1 = torch.tensor(row1[params_cols].values[0], dtype=torch.float32)
                ACC1 = row1["Accuracy"].values[0]
                ind1 = row1.index[0]
                Stream2 = torch.tensor(row2[params_cols].values[0], dtype=torch.float32)
                ACC2 = row2["Accuracy"].values[0]
                ind2 = row2.index[0]
                target = torch.tensor(row3[params_cols].values[0], dtype=torch.float32)
                ACC3 = row3["Accuracy"].values[0]
                ind3 = row3.index[0]
                loaded = torch.stack([Stream1, Stream2, target], dim=0)
                L_ACC_test.append([ACC1, ACC2, ACC3])
                L_indexes_test.append([ind1, ind2, ind3])
                L_loaded_test.append(loaded)
        
        if L_loaded_test:
            torch.save({"loaded_list": L_loaded_test, "L_ACC_list": L_ACC_test, "L_indexes_list": L_indexes_test}, 
                      os.path.join(results_path, "test_batches.pt"))
            test_batches = len(L_loaded_test)
        
        del L_loaded_test, L_ACC_test, L_indexes_test
        gc.collect()
    
    return train_batches, val_batches, test_batches

# Main processing loop
for m in [0, 1, 2]:
    for activ_key in tqdm(range(6), desc=f"Activations (m={m})", leave=False):
        for epoch_key in tqdm(range(6), desc=f"Epochs (m={m}, a={activ_key})", leave=False):
            total_scenarios += 1
            train_b, val_b, test_b = process_scenario_in_chunks(m, activ_key, epoch_key, chunk_size=100)
            
            if train_b > 0 or val_b > 0 or test_b > 0:
                processed_scenarios += 1
                print(f"Completed overlapping_m{m}_epoch{epoch_key}_activ{activ_key}: train={train_b}, val={val_b}, test={test_b}")

print(f"\nConversion complete: {processed_scenarios}/{total_scenarios} scenarios processed")
print("Ready to run cell #6 to create clean merged scenarios")

Converting .npy files to .pth files for all overlapping scenarios...


Epochs (m=0, a=0):   0%|          | 0/6 [00:00<?, ?it/s]

Processing overlapping_m0_epoch0_activ0 - 11076 train pairs...




Train chunks m0e0a0:   0%|          | 0/111 [00:00<?, ?it/s]

Train chunks m0e0a0:   1%|          | 1/111 [00:00<01:21,  1.35it/s]

Train chunks m0e0a0:   2%|         | 2/111 [00:01<01:21,  1.34it/s]

Train chunks m0e0a0:   3%|         | 3/111 [00:02<01:19,  1.36it/s]

Train chunks m0e0a0:   4%|         | 4/111 [00:02<01:18,  1.36it/s]

Train chunks m0e0a0:   5%|         | 5/111 [00:03<01:17,  1.37it/s]

Train chunks m0e0a0:   5%|         | 6/111 [00:04<01:16,  1.38it/s]

Train chunks m0e0a0:   6%|         | 7/111 [00:05<01:15,  1.38it/s]

Train chunks m0e0a0:   7%|         | 8/111 [00:05<01:14,  1.37it/s]

Train chunks m0e0a0:   8%|         | 9/111 [00:06<01:14,  1.37it/s]

Train chunks m0e0a0:   9%|         | 10/111 [00:07<01:13,  1.37it/s]

Train chunks m0e0a0:  10%|         | 11/111 [00:08<01:13,  1.37it/s]

Train chunks m0e0a0:  11%|         | 12/111 [00:08<01:12,  1.37it/s]

Train chunks m0e0a0:  12%|        | 13/111 [00:09<01:11,  1.37it/s]

Train chunks m0e0a0:  13%|        |

Merging 111 train chunks...


...

Completed overlapping_m1_epoch5_activ4: train=44496, val=9954, test=132150



Epochs (m=1, a=5):   0%|          | 0/6 [00:00<?, ?it/s]

Processing overlapping_m1_epoch0_activ5 - 44496 train pairs...




Train chunks m1e0a5:   0%|          | 0/445 [00:00<?, ?it/s]

Train chunks m1e0a5:   0%|          | 1/445 [00:00<05:25,  1.36it/s]

Train chunks m1e0a5:   0%|          | 2/445 [00:01<05:24,  1.37it/s]

Train chunks m1e0a5:   1%|          | 3/445 [00:02<05:22,  1.37it/s]

Train chunks m1e0a5:   1%|          | 4/445 [00:02<05:22,  1.37it/s]

Train chunks m1e0a5:   1%|          | 5/445 [00:03<05:25,  1.35it/s]

Train chunks m1e0a5:   1%|         | 6/445 [00:04<05:27,  1.34it/s]

Train chunks m1e0a5:   2%|         | 7/445 [00:05<05:24,  1.35it/s]

Train chunks m1e0a5:   2%|         | 8/445 [00:05<05:22,  1.36it/s]

Train chunks m1e0a5:   2%|         | 9/445 [00:06<05:20,  1.36it/s]

Train chunks m1e0a5:   2%|         | 10/445 [00:07<05:24,  1.34it/s]

Train chunks m1e0a5:   2%|         | 11/445 [00:08<05:26,  1.33it/s]

Train chunks m1e0a5:   3%|         | 12/445 [00:08<05:28,  1.32it/s]

Train chunks m1e0a5:   3%|         | 13/445 [00:09<05:24,  1.33it/s]

Train chunks m1e0a5:   3%|    

RuntimeError: [enforce fail at inline_container.cc:659] . unexpected pos 1295168 vs 1295056

In [2]:
!pip show torch

Name: torch
Version: 2.7.1+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /home/aymen/anaconda3/envs/FCL/lib/python3.10/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, bitsandbytes, geomloss, peft, pytorch-ranger, torch-optimizer, torchaudio, torchvision


([], [])

In [ ]:
train_pairs[0]

In [4]:
# Cell: Combine Existing .pth Files by Overlap Level
import torch
import os
from tqdm import tqdm
import random

# Create new clean scenarios directory
scenarios_path = "./data/Scenarios"
if not os.path.exists(scenarios_path):
    os.makedirs(scenarios_path)

print("Combining existing .pth files by overlap level...")

# Define the 4 clean scenarios
scenarios = [
    {"name": "no_overlap", "m": 0, "description": "Zero overlap between digit sets"},
    {"name": "single_overlap", "m": 1, "description": "Exactly one digit overlap"}, 
    {"name": "double_overlap", "m": 2, "description": "Exactly two digits overlap"},
    {"name": "mixed_overlap", "m": "mixed", "description": "Combined and shuffled overlaps (0,1,2)"}
]

# Get all existing scenario folders and filter only our overlapping ones
base_scenario_path = "./data/Scenario"
all_existing_folders = [f for f in os.listdir(base_scenario_path) if os.path.isdir(os.path.join(base_scenario_path, f))]

# Filter for only our overlapping_m*_epoch*_activ* pattern
our_overlapping_folders = [f for f in all_existing_folders if f.startswith("overlapping_m") and "_epoch" in f and "_activ" in f]

print(f"Found {len(our_overlapping_folders)} overlapping scenario folders out of {len(all_existing_folders)} total folders")
print(f"Ignoring {len(all_existing_folders) - len(our_overlapping_folders)} non-overlapping folders")

for scenario in scenarios:
    print(f"\nProcessing: {scenario['name']} - {scenario['description']}")
    
    # Create scenario directory
    scenario_path = os.path.join(scenarios_path, scenario['name'])
    if not os.path.exists(scenario_path):
        os.makedirs(scenario_path)
    
    # Initialize merged data containers
    merged_train_loaded = []
    merged_train_acc = []
    merged_train_indexes = []
    
    merged_val_loaded = []
    merged_val_acc = []
    merged_val_indexes = []
    
    merged_test_loaded = []
    merged_test_acc = []
    merged_test_indexes = []
    
    # Collect paths to process based on our specific overlapping folders
    paths_to_process = []
    
    if scenario['m'] == 'mixed':
        # For mixed, collect all our overlapping folders
        for folder in our_overlapping_folders:
            path = os.path.join(base_scenario_path, folder)
            if os.path.exists(path):
                paths_to_process.append(path)
    else:
        # For specific overlap level, filter by m value
        m = scenario['m']
        for folder in our_overlapping_folders:
            if folder.startswith(f"overlapping_m{m}_"):
                path = os.path.join(base_scenario_path, folder)
                if os.path.exists(path):
                    paths_to_process.append(path)
    
    print(f"Processing {len(paths_to_process)} matching overlapping folders...")
    
    # Process each path and combine data
    for path in tqdm(paths_to_process, desc="Combining batches"):
        # Load train data
        train_file = os.path.join(path, "train_batches.pt")
        if os.path.exists(train_file):
            # FIXED: Add weights_only=False to avoid PyTorch 2.6+ error
            train_data = torch.load(train_file, map_location='cpu', weights_only=False)
            merged_train_loaded.extend(train_data['loaded_list'])
            merged_train_acc.extend(train_data['L_ACC_list'])
            merged_train_indexes.extend(train_data['L_indexes_list'])
        
        # Load val data
        val_file = os.path.join(path, "val_batches.pt")
        if os.path.exists(val_file):
            # FIXED: Add weights_only=False to avoid PyTorch 2.6+ error
            val_data = torch.load(val_file, map_location='cpu', weights_only=False)
            merged_val_loaded.extend(val_data['loaded_list'])
            merged_val_acc.extend(val_data['L_ACC_list'])
            merged_val_indexes.extend(val_data['L_indexes_list'])
        
        # Load test data
        test_file = os.path.join(path, "test_batches.pt")
        if os.path.exists(test_file):
            # FIXED: Add weights_only=False to avoid PyTorch 2.6+ error
            test_data = torch.load(test_file, map_location='cpu', weights_only=False)
            merged_test_loaded.extend(test_data['loaded_list'])
            merged_test_acc.extend(test_data['L_ACC_list'])
            merged_test_indexes.extend(test_data['L_indexes_list'])
    
    # For mixed scenario, shuffle the combined data
    if scenario['m'] == 'mixed':
        print("Shuffling mixed overlap data...")
        
        # Create shuffled indices
        train_indices = list(range(len(merged_train_loaded)))
        val_indices = list(range(len(merged_val_loaded)))
        test_indices = list(range(len(merged_test_loaded)))
        
        random.shuffle(train_indices)
        random.shuffle(val_indices)
        random.shuffle(test_indices)
        
        # Apply shuffle
        merged_train_loaded = [merged_train_loaded[i] for i in train_indices]
        merged_train_acc = [merged_train_acc[i] for i in train_indices]
        merged_train_indexes = [merged_train_indexes[i] for i in train_indices]
        
        merged_val_loaded = [merged_val_loaded[i] for i in val_indices]
        merged_val_acc = [merged_val_acc[i] for i in val_indices]
        merged_val_indexes = [merged_val_indexes[i] for i in val_indices]
        
        merged_test_loaded = [merged_test_loaded[i] for i in test_indices]
        merged_test_acc = [merged_test_acc[i] for i in test_indices]
        merged_test_indexes = [merged_test_indexes[i] for i in test_indices]
    
    print(f"Final counts:")
    print(f"  Train: {len(merged_train_loaded)} batches")
    print(f"  Val: {len(merged_val_loaded)} batches")
    print(f"  Test: {len(merged_test_loaded)} batches")
    
    # Save merged data
    if merged_train_loaded:
        torch.save({
            "loaded_list": merged_train_loaded,
            "L_ACC_list": merged_train_acc,
            "L_indexes_list": merged_train_indexes,
            "scenario_info": {
                "name": scenario['name'],
                "description": scenario['description'],
                "overlap_level": scenario['m'],
                "source_folders_count": len(paths_to_process),
                "source_folders": [os.path.basename(p) for p in paths_to_process],
                "total_merged_pairs": len(merged_train_loaded) + len(merged_val_loaded) + len(merged_test_loaded)
            }
        }, os.path.join(scenario_path, "train_batches.pth"))
    
    if merged_val_loaded:
        torch.save({
            "loaded_list": merged_val_loaded,
            "L_ACC_list": merged_val_acc,
            "L_indexes_list": merged_val_indexes
        }, os.path.join(scenario_path, "val_batches.pth"))
    
    if merged_test_loaded:
        torch.save({
            "loaded_list": merged_test_loaded,
            "L_ACC_list": merged_test_acc,
            "L_indexes_list": merged_test_indexes
        }, os.path.join(scenario_path, "test_batches.pth"))

print("\n Successfully created 4 clean scenario folders in ./data/Scenarios")
print("\nFinal structure:")
for scenario in scenarios:
    scenario_path = os.path.join(scenarios_path, scenario['name'])
    if os.path.exists(scenario_path):
        files = os.listdir(scenario_path)
        print(f"  {scenario['name']}/: {len(files)} files ({', '.join(files)})")

Combining existing .pth files by overlap level...
Found 108 overlapping scenario folders out of 360 total folders
Ignoring 252 non-overlapping folders

Processing: no_overlap - Zero overlap between digit sets
Processing 36 matching overlapping folders...


Combining batches: 100%|| 36/36 [00:00<00:00, 60811.50it/s]


Final counts:
  Train: 0 batches
  Val: 0 batches
  Test: 0 batches

Processing: single_overlap - Exactly one digit overlap
Processing 36 matching overlapping folders...


Combining batches: 100%|| 36/36 [00:00<00:00, 29829.11it/s]


Final counts:
  Train: 0 batches
  Val: 0 batches
  Test: 0 batches

Processing: double_overlap - Exactly two digits overlap
Processing 36 matching overlapping folders...


Combining batches: 100%|| 36/36 [00:00<00:00, 28371.84it/s]


Final counts:
  Train: 0 batches
  Val: 0 batches
  Test: 0 batches

Processing: mixed_overlap - Combined and shuffled overlaps (0,1,2)
Processing 108 matching overlapping folders...


Combining batches: 100%|| 108/108 [00:00<00:00, 68457.73it/s]

Shuffling mixed overlap data...
Final counts:
  Train: 0 batches
  Val: 0 batches
  Test: 0 batches

 Successfully created 4 clean scenario folders in ./data/Scenarios

Final structure:
  no_overlap/: 0 files ()
  single_overlap/: 0 files ()
  double_overlap/: 0 files ()
  mixed_overlap/: 0 files ()
